In [ ]:
import pandas as pd
import pandas_ta as ta
import pyotp
from datetime import datetime
import time
import csv
import logging
import websocket as ws_client
import threading

from NorenRestApiPy.NorenApi import NorenApi

class ShoonyaApiPy(NorenApi):
    def __init__(self):
        NorenApi.__init__(self, host='https://api.shoonya.com/NorenWClientTP/', websocket='wss://api.shoonya.com/NorenWSTP/')        
        global api
        api = self

import logging
#logging.basicConfig(level=logging.DEBUG)
logging.getLogger('websocket').setLevel(logging.INFO)


usercred = pd.read_excel("usercred.xlsx")
user    = usercred.iloc[0,0]
pwd     = usercred.iloc[0,1]
vc      = usercred.iloc[0,2]
app_key = usercred.iloc[0,3]
imei    = usercred.iloc[0,4]
qr = usercred.iloc[0,5]
factor2 = pyotp.TOTP(qr).now()

api = ShoonyaApiPy()
ret = api.login(userid=user, password=pwd, twoFA=factor2, vendor_code=vc, api_secret=app_key, imei=imei)

In [ ]:
############## TRADE LOGIC with all moduler function
import threading
import time
from datetime import datetime
import pandas as pd
import numpy as np
import csv

# Global flag
feed_opened = False
feedJson = {}
data_updated = False  
last_resampled_data = {}
sample_count = 0

# Define global variables and constants
current_positions = {}
ema_length = 20
initial_sl_points = 20
move_sl_points = 20
profit_booking_points = 100

position_lock = threading.Lock()

def event_handler_feed_update(tick_data):
    try:
        if 'lp' in tick_data and 'tk' in tick_data:
            timest = datetime.fromtimestamp(int(tick_data['ft'])).isoformat()
            token = tick_data['tk']            
            # Check if token exists in feedJson
            if token not in feedJson:
                feedJson[token] = []  # Create an empty list for new tokens

            # Append the new tick data to the list
            feedJson[token].append({'ltp': float(tick_data['lp']), 'tt': timest})
    
    except (KeyError, ValueError) as e:
        print(f"Error processing tick data: {e}")  # Informative error message
    
def event_handler_order_update(tick_data):
    print(f"Order update {tick_data}")

def open_callback():
    global feed_opened
    feed_opened = True

# Assuming api.start_websocket(...) is correctly initialized elsewhere
api.start_websocket(order_update_callback=event_handler_order_update,
                    subscribe_callback=event_handler_feed_update, 
                    socket_open_callback=open_callback)

while not feed_opened:
    pass

# Subscribe to multiple tokens
api.subscribe(['NSE|26009'])

# Event to signal thread to stop
stop_event = threading.Event()

def append_to_csv(trade_log):
    with open('trading_log.csv', 'a', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(trade_log)

def resample_data(stop_event):
    global data_updated
    global last_resampled_data
        
    while not stop_event.is_set():
        try:
            new_data_available = False            
            current_feed_json = {**feedJson}            

            for token, data_list in current_feed_json.items():
                df_token = pd.DataFrame(data_list)
                df_token['tt'] = pd.to_datetime(df_token['tt'])  # Convert 'tt' column to datetime if needed                

                df_token.set_index('tt', inplace=True)                
                df_token.dropna(subset=['ltp'], inplace=True)                                  
                ############ Resample and aggregate OHLC
                df_resampled = df_token['ltp'].resample('15s').ohlc()       
                      
                # Handle NaN values if any
                df_resampled.dropna(inplace=True) 
                df_resampled['ema1'] = df_resampled['close'].ewm(span=3, adjust=False).mean().round(2)
                df_resampled['ema2'] = df_resampled['close'].ewm(span=10, adjust=False).mean().round(2)
                df_resampled['ema3'] = df_resampled['close'].ewm(span=20, adjust=False).mean().round(2)
               
                # Convert to Series to avoid length mismatch
                df_resampled_series = df_resampled.apply(pd.Series)                    

                if token not in last_resampled_data:
                    last_resampled_data[token] = df_resampled_series
                    new_data_available = True  # Set flag to indicate new data is available
                else:
                    # Concatenate new data to the existing DataFrame                    
                    combined_df = pd.concat([last_resampled_data[token].dropna(axis=1, how='all'), 
                             df_resampled_series.dropna(axis=1, how='all')])
                    
                    # Drop any duplicate rows based on the index (timestamp)
                    combined_df = combined_df[~combined_df.index.duplicated(keep='last')]
                    
                    # Update last_resampled_data with the de-duplicated data
                    last_resampled_data[token] = combined_df
                    new_data_available = True        
                                                 
            if new_data_available:
                data_updated = True  # Set the flag only if new data is available
            time.sleep(1)  # Adjust interval as needed for checking new data

        except ValueError as ve:
            print(f"ValueError in resample_data: {ve}")  # Handle specific ValueError
        
        except Exception as e:
            print(f"Error in resample_data: {e}")  # Error handling

def trading_logic(stop_event):
    global data_updated
    global last_resampled_data
    current_positions = {}
    ema_length = 20
    
    while not stop_event.is_set():
        try:
            if data_updated:
                data_updated = False
                for token, df in last_resampled_data.items():
                    process_token_data(token, df)
        except Exception as e:
            print(f"Error in trading logic: {e}")


# Start the resampling thread
thread_resample = threading.Thread(target=resample_data, args=(stop_event,), daemon=True)
thread_resample.start()

# Start the trading logic thread
thread_trade = threading.Thread(target=trading_logic, args=(stop_event,), daemon=True)
thread_trade.start()

In [ ]:
# Define the new functions for entry and exit conditions
def check_long_entry_conditions(df, token):
    ema1_latest = df['ema1'].iloc[-1]
    ema2_latest = df['ema2'].iloc[-1]
    ema3_latest = df['ema3'].iloc[-1]
    ema1_previous = df['ema1'].iloc[-2]
    ema2_previous = df['ema2'].iloc[-2]
    
    if (len(df)> ema_length) and ema1_latest > ema2_latest and ema1_previous <= ema2_previous and ema1_latest >= ema3_latest and ema2_latest >= ema3_latest:
        if current_positions.get(token) != 'long':
            current_positions[token] = {
                'position': 'long',
                'entry_price': df['close'].iloc[-1],
                'entry_time': df.index[-1],
                'initial_sl': df['close'].iloc[-1] - initial_sl_points,
                'sl_moved': False
            }
            print(f"Enter Long: {token} at {current_positions[token]['entry_time']}, Price: {current_positions[token]['entry_price']}")
            append_to_csv(["Enter Long", token, current_positions[token]['entry_time'], current_positions[token]['entry_price']])

def check_long_exit_conditions(df, token, current_price):
    entry_price = current_positions[token]['entry_price']
    initial_sl = current_positions[token]['initial_sl']
    sl_moved = current_positions[token]['sl_moved']
    
    if not sl_moved and current_price >= entry_price + move_sl_points:
        current_positions[token]['initial_sl'] = entry_price
        current_positions[token]['sl_moved'] = True
        print(f"SL moved to entry price for long position on {token}")

    if current_price <= initial_sl or current_price >= entry_price + profit_booking_points or (df['ema1'].iloc[-1] < df['ema2'].iloc[-1] and df['ema1'].iloc[-2] > df['ema2'].iloc[-2]):
        exit_timestamp = df.index[-1]
        exit_price = current_price
        print(f"Exit Long: {token} at {exit_timestamp}, Price: {exit_price}")
        append_to_csv(["Exit Long", token, exit_timestamp, exit_price])

def check_short_entry_conditions(df, token):
    ema1_latest = df['ema1'].iloc[-1]
    ema2_latest = df['ema2'].iloc[-1]
    ema3_latest = df['ema3'].iloc[-1]
    ema1_previous = df['ema1'].iloc[-2]
    ema2_previous = df['ema2'].iloc[-2]
    
    if (len(df)> ema_length) and ema1_latest < ema2_latest and ema1_previous >= ema2_previous and ema1_latest <= ema3_latest and ema2_latest <= ema3_latest:
        if current_positions.get(token) != 'short':
            current_positions[token] = {
                'position': 'short',
                'entry_price': df['close'].iloc[-1],
                'entry_time': df.index[-1],
                'initial_sl': df['close'].iloc[-1] + initial_sl_points,
                'sl_moved': False
            }
            print(f"Enter Short: {token} at {current_positions[token]['entry_time']}, Price: {current_positions[token]['entry_price']}")
            append_to_csv(["Enter Short", token, current_positions[token]['entry_time'], current_positions[token]['entry_price']])

def check_short_exit_conditions(df, token, current_price):
    entry_price = current_positions[token]['entry_price']
    initial_sl = current_positions[token]['initial_sl']
    sl_moved = current_positions[token]['sl_moved']
    
    if not sl_moved and current_price <= entry_price - move_sl_points:
        current_positions[token]['initial_sl'] = entry_price
        current_positions[token]['sl_moved'] = True
        print(f"SL moved to entry price for short position on {token}")

    if current_price >= initial_sl or current_price <= entry_price - profit_booking_points or (df['ema1'].iloc[-1] > df['ema2'].iloc[-1] and df['ema1'].iloc[-2] < df['ema2'].iloc[-2]):
        exit_timestamp = df.index[-1]
        exit_price = current_price
        print(f"Exit Short: {token} at {exit_timestamp}, Price: {exit_price}")
        append_to_csv(["Exit Short", token, exit_timestamp, exit_price])
        current_positions[token] = None


def process_token_data(token, df):
    current_feed_json = {**feedJson}
    if len(df) > ema_length and 'ema1' in df.columns and 'ema2' in df.columns and 'ema3' in df.columns:
        
        check_long_entry_conditions(df, token)
        
        if current_positions.get(token) and current_positions[token]['position'] == 'long':
            current_price = current_feed_json.get(token, {}).get('ltp')
            check_long_exit_conditions(df, token, current_price)
        
        check_short_entry_conditions(df, token)
        
        if current_positions.get(token) and current_positions[token]['position'] == 'short':
            current_price = current_feed_json.get(token, {}).get('ltp')
            check_short_exit_conditions(df, token, current_price)

In [ ]:
last_resampled_data

In [ ]:
api.close_websocket()
# Stop the thread gracefully by setting the event
stop_event.set()
print("Thread stopped gracefully")

In [ ]:
import csv

# Assuming last_resampled_data is defined and populated with data

# Specify the CSV file path
csv_file = 'last_resampled_data1.csv'

try:
    # Open the CSV file in write mode
    with open(csv_file, 'w', newline='') as file:
        writer = csv.writer(file)

        # Write headers for each token's data
        writer.writerow(['Token', 'Timestamp', 'Open', 'High', 'Low', 'Close', 'EMA1', 'EMA2' , 'EMA3'])

        # Iterate through each token and its corresponding DataFrame
        for token, df in last_resampled_data.items():
            # Extract relevant columns for writing to CSV
            for index, row in df.iterrows():
                writer.writerow([token, index, row['open'], row['high'], row['low'], row['close'], row['ema1'], row['ema2'], row['ema3']])

    print(f"Data successfully written to {csv_file}")

except Exception as e:
    print(f"Error writing data to CSV: {e}")
